# Effects and handlers

Gallowglass tracks effects in type signatures. A pure function has an empty effect row; an I/O-performing function carries `IO` in its row; an exception-raising function carries `Exn`. The compiler discharges effects through `handle` expressions.

This lesson covers the surface syntax — declarations, signatures, the `handle` shape — plus the design rule that makes the system honest. The runtime story has one wrinkle the kernel doesn't expose interactively yet (running a CPS-encoded handler result needs a helper the typechecker doesn't know about); that's noted explicitly below.

This lesson assumes you've worked through the previous three.

## Pure functions: the empty row

A function with no annotated effect row is pure. The kernel's decl-summary shows the inferred type without a row when the row is empty:

In [1]:
let double : Nat → Nat = λ n → n + n

double : Nat → Nat

## Effectful signatures

Effect rows appear in curly braces between the arrow and the result type: `Nat → {Counter} Nat` is a function from `Nat` to `Nat` that performs `Counter` effects on the way.

Define an effect with `eff`:

In [2]:
eff Counter {
  inc : Nat → Nat
}

Now functions that perform `Counter.inc` carry it in their row. The decl summary shows the row as part of the type:

In [3]:
let bumped : Nat → {Counter} Nat = λ n → inc n

bumped : Nat → {Counter} Nat

## The `handle` form

`handle` discharges an effect locally. The shape is:

```
handle <expression> {
  | return xx       → <pure-result branch>
  | <op_name> arg k → <continuation branch>
}
```

`return xx` is the arm that fires when the inner expression evaluates to a pure value `xx`. Each effect-op arm receives the operation's argument plus a `continuation` `k` — calling `k value` resumes the computation with `value` as the op's result.

Define a handler that intercepts `inc`:

In [4]:
let result = handle (inc 0) {
  | return xx → xx
  | inc _ kk  → kk 42
}

result : ∀ a. {∅ | a} Nat

Notice the inferred type — `result` has shape `{∅ | r} Nat`, a CPS-encoded computation that, when run, would discharge the `Counter` effect and produce `42`.

**Wrinkle:** the kernel doesn't currently expose the `run` helper that takes a CPS computation and produces its underlying value. So evaluating `result` directly in a cell shows the unrun computation as a Law, not the integer it would compute to. The `tests/bootstrap/test_codegen.py` suite exercises `run` via a Python helper; making it accessible from cells is forward work.

You can still see the result's structure:

In [5]:
result

<law arity=2 name='result_handle'>

## The `External` effect — VM boundaries

Operations that cross the VM boundary (filesystem, process I/O, foreign calls) carry the `External` effect. `external mod` declarations register the type and effect of each operation; the bootstrap registers them as Pin'd Laws that delegate to the underlying BPLAN or RPLAN op.

The pre-existing `Reaver.RPLAN` declarations look like this (no need to re-declare; this is illustrative):

```gallowglass
external mod Reaver.RPLAN {
  output : Nat → Nat
  input  : Nat → Nat
}
```

A function calling these operations would carry the effect in its row — typically `{External | r}` so the caller can extend with their own effects.

## The `Abort` invariant

`Abort` is **not** in any effect row. It's the unhandle-able effect that propagates straight to the VM's virtualization supervisor — the runtime equivalent of "this computation cannot continue." Because it's never in a row, it can't be handled by user code, and the compiler enforces this at the row-typing level.

You won't encounter `Abort` directly when writing Gallowglass — it surfaces only when the runtime has to give up (out-of-memory, unrecoverable invariant violation, etc.). It exists in the design as a hard floor so handler logic stays focused on recoverable effects.

## What you've seen and where it goes from here

Effect rows give the type signature *visible* — anyone reading `read_file : Path → {IO, Exn IOError | r} Bytes` knows what side-effects the call performs without checking the implementation. Handlers discharge those effects locally with explicit continuation control.

For the full reference, see `doc/language-guide.md` sections on the effect system and `spec/05-type-system.md` for the formal row-typing rules. The `tests/bootstrap/test_codegen.py::test_eff_*` tests are the most compact illustration of the CPS encoding the bootstrap uses, including the `run` helper that's missing from the kernel today.

You've now worked through the four core notebooks: declarations and pattern matching (lesson 1), typeclasses (lesson 2), Glass IR (lesson 3), and effects (this one). Together they cover everything you need to read most Gallowglass source. For the LLM-shaped condensed reference, see `doc/phrasebook.md`.